# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [8]:
# Write your code below.
%load_ext dotenv
%dotenv


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [9]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [10]:
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")

parquet_files = glob(
    os.path.join(PRICE_DATA, "**/*.parquet"), recursive=True
)


In [15]:
print(len(parquet_files)) 
print(parquet_files[:5])

import pandas as pd
pd.read_parquet(parquet_files[0]).head()

3118
['../../05_src/data/prices/NQP/NQP_2014/part.0.parquet', '../../05_src/data/prices/NQP/NQP_2014/part.1.parquet', '../../05_src/data/prices/NQP/NQP_2013/part.0.parquet', '../../05_src/data/prices/NQP/NQP_2013/part.1.parquet', '../../05_src/data/prices/NQP/NQP_2012/part.0.parquet']


,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year
__null_dask_index__,,,,,,,,,,
5761,2014-01-02,12.39,12.45,12.35,12.38,8.804968,78900,NQP.csv,NQP,2014
5762,2014-01-03,12.44,12.52,12.40,12.52,8.904537,22800,NQP.csv,NQP,2014
5763,2014-01-06,12.58,12.66,12.56,12.60,8.961439,46900,NQP.csv,NQP,2014
5764,2014-01-07,12.62,12.70,12.62,12.70,9.032561,63200,NQP.csv,NQP,2014
5765,2014-01-08,12.64,12.70,12.62,12.67,9.011220,45500,NQP.csv,NQP,2014


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [12]:
# Write your code below.
import dask.dataframe as dd

dd_px = dd.read_parquet(parquet_files)

# Match the column name used in the assignment.
dd_px = dd_px.rename(columns={"Adj Close": "Adj_Close"})

dd_px = dd_px.set_index("ticker")


def add_features(df):
    df = df.sort_values("Date")

    df["Close_lag_1"] = df["Close"].shift(1)
    df["Adj_Close_lag_1"] = df["Adj_Close"].shift(1)

    df["returns"] = df["Close"] / df["Close_lag_1"] - 1
    df["hi_lo_range"] = df["High"] - df["Low"]

    return df


dd_feat = dd_px.groupby("ticker", group_keys=False).apply(
    add_features,
    meta=add_features(dd_px._meta)
)



+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [13]:
# Write your code below.
df_feat = dd_feat.compute()

df_feat = df_feat.reset_index()
df_feat = df_feat.sort_values(["ticker", "Date"])


def moving_average(returns):
    return returns.rolling(10).mean()


df_feat["returns_ma_10"] = (
    df_feat.groupby("ticker")["returns"].transform(moving_average)
)

df_feat.head()



,ticker,Date,Open,High,Low,Close,Adj_Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,returns_ma_10
0,A,1999-11-18,32.546494,35.765381,28.612303,31.473534,27.068665,62546300.0,A.csv,1999,NaN,NaN,NaN,7.153078,NaN
1,A,1999-11-19,30.713520,30.758226,28.478184,28.880543,24.838577,15234100.0,A.csv,1999,31.473534,27.068665,-0.082386,2.280043,NaN
2,A,1999-11-22,29.551144,31.473534,28.657009,31.473534,27.068665,6577800.0,A.csv,1999,28.880543,24.838577,0.089783,2.816525,NaN
3,A,1999-11-23,30.400572,31.205294,28.612303,28.612303,24.607880,5975600.0,A.csv,1999,31.473534,27.068665,-0.090909,2.592991,NaN
4,A,1999-11-24,28.701717,29.998211,28.612303,29.372318,25.261524,4843200.0,A.csv,1999,28.612303,24.607880,0.026563,1.385908,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

No, converting to pandas was not necessary because Dask can also calculate moving averages.

Pandas is simpler but Dask is more helpful for larger datasets

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.